# Converting C1999 SMA to Geogrid specific binary format #

In [31]:
import xarray as xr
import numpy as np
import os
import matplotlib.pyplot as plt

In [32]:
file_clay = "C1999-SMA_CLAY_minfrac_0.083deg.nc"
file_silt = "C1999-SMA_SILT_minfrac_0.083deg.nc"

In [33]:
ds_clay = xr.open_dataset(file_clay)
ds_silt = xr.open_dataset(file_silt)

In [34]:


lat = ds_clay['latitude'].values
lon = ds_clay['longitude'].values

lat_max =  39.10337 
lat_min = -3.0628967 
lon_max = 66.38614 
lon_min = -26.386139




In [35]:
lat_name = "lat" if "lat" in ds_clay.coords else "latitude"
lon_name = "lon" if "lon" in ds_clay.coords else "longitude"

lat_clay = ds_clay[lat_name].values
lon_clay = ds_clay[lon_name].values
lat_silt = ds_silt[lat_name].values
lon_silt = ds_silt[lon_name].values

In [36]:
for var in ds_clay:
    print(f"Variable: {var}, shape: {ds_clay[var].shape}")

Variable: soiltype, shape: (2160, 4320)
Variable: illi, shape: (2160, 4320)
Variable: kaol, shape: (2160, 4320)
Variable: mont, shape: (2160, 4320)
Variable: calc, shape: (2160, 4320)
Variable: quar, shape: (2160, 4320)
Variable: feld, shape: (2160, 4320)
Variable: hema, shape: (2160, 4320)
Variable: gyps, shape: (2160, 4320)


In [37]:
for var in ds_silt:
    print(f"Variable: {var}, shape: {ds_silt[var].shape}")

Variable: soiltype, shape: (2160, 4320)
Variable: illi, shape: (2160, 4320)
Variable: kaol, shape: (2160, 4320)
Variable: mont, shape: (2160, 4320)
Variable: calc, shape: (2160, 4320)
Variable: quar, shape: (2160, 4320)
Variable: feld, shape: (2160, 4320)
Variable: hema, shape: (2160, 4320)
Variable: gyps, shape: (2160, 4320)


In [38]:
#value range of each mineral #


#mineral_vars_clay = [var for var in ds_clay.data_vars if var not in ds_clay.coords]

mineral_vars = ['illi', 'kaol', 'mont', 'calc','quar','feld','hema','gyps']

print ("CLAY FILE")
for var in mineral_vars:
    da = ds_clay[var]
    print(f"Variable: {var}, min: {da.min().values}, max: {da.max().values}")

print ("SILT FILE")
for var in mineral_vars:
    da = ds_silt[var]
    print(f"Variable: {var}, min: {da.min().values}, max: {da.max().values}")

CLAY FILE
Variable: illi, min: 0.08839469403028488, max: 0.5528103709220886
Variable: kaol, min: 0.035325486212968826, max: 0.6894786357879639
Variable: mont, min: 0.026518408209085464, max: 0.5398313999176025
Variable: calc, min: 0.0, max: 0.2561097741127014
Variable: quar, min: 0.008831371553242207, max: 0.13593560457229614
Variable: feld, min: 0.002020202111452818, max: 0.24752475321292877
Variable: hema, min: 0.0, max: 0.06930693238973618
Variable: gyps, min: 0.004999999888241291, max: 0.1607142835855484
SILT FILE
Variable: illi, min: 0.0, max: 0.0
Variable: kaol, min: 0.0, max: 0.0
Variable: mont, min: 0.0, max: 0.0
Variable: calc, min: 0.0, max: 0.6700000166893005
Variable: quar, min: 0.05000000074505806, max: 0.9108911156654358
Variable: feld, min: 0.009999999776482582, max: 0.7843137383460999
Variable: hema, min: 0.0, max: 0.06930693238973618
Variable: gyps, min: 0.009803921915590763, max: 0.25999999046325684


In [39]:
# sum on each grid cell#

clay_sum = sum(ds_clay[var] for var in mineral_vars)

In [40]:
slt_sum = sum(ds_silt[var] for var in mineral_vars)


In [ ]:
CLAY_FILE = "C1999-SMA_CLAY_minfrac_0.083deg.nc"
SILT_FILE = "C1999-SMA_SILT_minfrac_0.083deg.nc"

OUT_ROOT = "C1999_geog"

WORDSIZE = 4
PRECISION = 1e-6
MISSING_VALUE = -9999

MINERAL_VARS = ["illi", "kaol", "mont", "calc", "quar", "feld", "hema", "gyps"]

In [42]:
CLAY_VARS = ["illi", "kaol", "mont", "calc", "quar", "feld", "hema", "gyps"]
SILT_VARS = ["calc", "quar", "feld", "hema", "gyps"]  # In silt file, illi, kaol, mont are zero everywhere

In [ ]:
## WRITE INDEX  FILE ALONG WITH BINARY FILES FOR EACH MINERAL FRACTION#

def write_index(folder, dx, dy,
                known_lat, known_lon,
                tile_x, tile_y,
                description):

    index_text = f"""type=continuous
signed=yes
projection=regular_ll
dx={dx}
dy={dy}
known_x=1.0
known_y=1.0
known_lat={known_lat}
known_lon={known_lon}
wordsize={WORDSIZE}
tile_x={tile_x}
tile_y={tile_y}
tile_z=1
scale_factor={PRECISION}
# missing_value={MISSING_VALUE}
units="fraction"
description="{description}"
"""

    with open(os.path.join(folder, "index"), "w") as f:
        f.write(index_text)

In [44]:
# FUNCTION TO CONVERT NETCDF FILES TO BINARY FILES FOR EACH MINERAL FRACTION#

def convert_to_geogrid(ncfile, variables, prefix):

    ds = xr.open_dataset(ncfile)

    lat = ds["latitude"].values
    lon = ds["longitude"].values

    dy = abs(float(lat[1] - lat[0]))
    dx = abs(float(lon[1] - lon[0]))

    tile_y = len(lat)
    tile_x = len(lon)

    known_lat = float(np.min(lat))
    known_lon = float(np.min(lon))

    binary_name = f"00001-{tile_x:05d}.00001-{tile_y:05d}"

    print("\nReading:", ncfile)
    print("Grid size:", tile_y, "x", tile_x)
    print("dx:", dx)
    print("dy:", dy)
    print("known_lat:", known_lat)
    print("known_lon:", known_lon)

    for var in variables:

        print(f"\nConverting {prefix}_{var}")

        data = ds[var].values.astype(np.float64)

        # WPS expects south-to-north order.
        # If input latitude is north-to-south, flip it.
        if lat[0] > lat[-1]:
            data = np.flipud(data)

        valid = np.isfinite(data)

        scaled = np.empty(data.shape, dtype=np.int64)

        # Valid mineral fractions are scaled
        scaled[valid] = np.rint(
            data[valid] / PRECISION
        ).astype(np.int64)

        # Missing values are NOT scaled
        scaled[~valid] = MISSING_VALUE

        # Convert to big-endian 4-byte integer
        scaled = scaled.astype(">i4")

        folder_name = f"{prefix.lower()}_{var.lower()}"
        folder = os.path.join(OUT_ROOT, folder_name)
        os.makedirs(folder, exist_ok=True)

        binary_path = os.path.join(folder, binary_name)

        # Write binary file
        scaled.tofile(binary_path)

        # Write index file
        write_index(
            folder=folder,
            dx=dx,
            dy=dy,
            known_lat=known_lat,
            known_lon=known_lon,
            tile_x=tile_x,
            tile_y=tile_y,
            description=f"{prefix} {var} mineral fraction"
        )

        size_mb = os.path.getsize(binary_path) / 1024 / 1024
        print("Written:", binary_path)
        print("Size MB:", size_mb)

    ds.close()

In [45]:
# CONVERT CLAY AND SILT NETCDF FILES TO BINARY FILES FOR EACH MINERAL FRACTION#

convert_to_geogrid(CLAY_FILE,CLAY_VARS,"C1999_CLAY")
convert_to_geogrid(SILT_FILE,SILT_VARS,"C1999_SILT")

print("C1999 geogrid conversion completed")
print("Output directory:", OUT_ROOT)


Reading: C1999-SMA_CLAY_minfrac_0.083deg.nc
Grid size: 2160 x 4320
dx: 0.08333332999998788
dy: 0.08333333000000209
known_lat: -89.958333335
known_lon: -179.958333335

Converting C1999_CLAY_illi
Written: C1999_geogrid\c1999_clay_illi\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_kaol
Written: C1999_geogrid\c1999_clay_kaol\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_mont
Written: C1999_geogrid\c1999_clay_mont\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_calc
Written: C1999_geogrid\c1999_clay_calc\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_quar
Written: C1999_geogrid\c1999_clay_quar\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_feld
Written: C1999_geogrid\c1999_clay_feld\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_hema
Written: C1999_geogrid\c1999_clay_hema\00001-04320.00001-02160
Size MB: 35.595703125

Converting C1999_CLAY_gyps
Written: C1999

In [46]:
print(os.listdir("C1999_geogrid"))

['c1999_clay_calc', 'c1999_clay_feld', 'c1999_clay_gyps', 'c1999_clay_hema', 'c1999_clay_illi', 'c1999_clay_kaol', 'c1999_clay_mont', 'c1999_clay_quar', 'c1999_silt_calc', 'c1999_silt_feld', 'c1999_silt_gyps', 'c1999_silt_hema', 'c1999_silt_quar']
